# Telegram GitHub Photo Publisher

Reads images from this repository's `images/` folder, chooses a random caption from `comments.txt`, posts to Telegram, and optionally deletes the previous bot-posted photo.

Required environment variables: `TELEGRAM_BOT_TOKEN`, `TELEGRAM_CHAT_ID`.
Never put your actual bot token in this notebook or a public repository.


In [ ]:
import os
import json
import random
import mimetypes
from pathlib import Path
from datetime import datetime, timezone

import requests

REPOSITORY = os.getenv("GITHUB_REPOSITORY", "YOUR_GITHUB_USERNAME/YOUR_REPOSITORY")
BRANCH = os.getenv("GITHUB_REF_NAME", "main")
BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "").strip()
CHAT_ID = os.getenv("TELEGRAM_CHAT_ID", "").strip()

STATE_FILE = Path("state.json")
DELETE_PREVIOUS_POST = True
MAX_IMAGE_BYTES = 10 * 1024 * 1024

if not BOT_TOKEN:
    raise RuntimeError("Missing TELEGRAM_BOT_TOKEN environment variable.")
if not CHAT_ID:
    raise RuntimeError("Missing TELEGRAM_CHAT_ID environment variable.")
if "/" not in REPOSITORY or REPOSITORY.startswith("YOUR_"):
    raise RuntimeError("Set GITHUB_REPOSITORY to 'owner/repo' or run through GitHub Actions.")

OWNER, REPO = REPOSITORY.split("/", 1)
API_BASE = f"https://api.telegram.org/bot{BOT_TOKEN}"
GITHUB_API = f"https://api.github.com/repos/{OWNER}/{REPO}"

print(f"Repository: {REPOSITORY}")
print(f"Branch: {BRANCH}")
print(f"Target Telegram chat: {CHAT_ID}")


In [ ]:
def telegram_call(method, *, data=None, files=None, timeout=60):
    url = f"{API_BASE}/{method}"

    try:
        response = requests.post(
            url,
            data=data,
            files=files,
            timeout=timeout
        )

        try:
            payload = response.json()
        except ValueError:
            raise RuntimeError(
                f"Telegram returned HTTP {response.status_code}. "
                f"Response: {response.text[:1000]}"
            )

        if not response.ok or not payload.get("ok"):
            error_code = payload.get("error_code", response.status_code)
            description = payload.get("description", "Unknown Telegram API error")
            raise RuntimeError(f"TELEGRAM ERROR {error_code}: {description}")

        return payload["result"]

    except requests.exceptions.Timeout:
        raise RuntimeError("Telegram request timed out.")
    except requests.exceptions.ConnectionError:
        raise RuntimeError("Could not connect to Telegram.")


def load_state():
    if not STATE_FILE.exists():
        return {}
    try:
        return json.loads(STATE_FILE.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return {}


def save_state(state):
    STATE_FILE.write_text(
        json.dumps(state, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8"
    )


def list_repository_images():
    url = f"{GITHUB_API}/contents/images"
    headers = {"Accept": "application/vnd.github+json"}
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    allowed_extensions = {".jpg", ".jpeg", ".png", ".webp"}
    items = response.json()

    return [
        item for item in items
        if item.get("type") == "file"
        and Path(item.get("name", "")).suffix.lower() in allowed_extensions
        and item.get("download_url")
    ]


def load_comments():
    url = (
        f"https://raw.githubusercontent.com/"
        f"{OWNER}/{REPO}/{BRANCH}/comments.txt"
    )
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    comments = [
        line.strip()
        for line in response.text.splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    ]

    if not comments:
        raise RuntimeError("comments.txt is empty. Add at least one caption.")

    return comments


In [ ]:
def publish_random_photo():
    state = load_state()
    images = list_repository_images()

    if not images:
        raise RuntimeError("No supported images found in images/.")

    previous_name = state.get("last_image")
    candidates = [item for item in images if item["name"] != previous_name]

    if not candidates:
        candidates = images

    chosen = random.choice(candidates)
    caption = random.choice(load_comments())

    image_response = requests.get(chosen["download_url"], timeout=60)
    image_response.raise_for_status()
    image_bytes = image_response.content

    if len(image_bytes) > MAX_IMAGE_BYTES:
        raise RuntimeError(
            f"{chosen['name']} exceeds 10 MB. Resize or compress it first."
        )

    mime_type = mimetypes.guess_type(chosen["name"])[0] or "application/octet-stream"

    result = telegram_call(
        "sendPhoto",
        data={
            "chat_id": CHAT_ID,
            "caption": caption[:1024]
        },
        files={
            "photo": (chosen["name"], image_bytes, mime_type)
        }
    )

    new_message_id = result["message_id"]
    old_message_id = state.get("last_message_id")

    if DELETE_PREVIOUS_POST and old_message_id:
        try:
            telegram_call(
                "deleteMessage",
                data={
                    "chat_id": CHAT_ID,
                    "message_id": old_message_id
                }
            )
            print(f"Deleted previous Telegram message: {old_message_id}")
        except Exception as exc:
            print(f"Could not delete previous message {old_message_id}: {exc}")

    state.update({
        "last_image": chosen["name"],
        "last_message_id": new_message_id,
        "chat_id": str(CHAT_ID),
        "last_caption": caption[:1024],
        "posted_at_utc": datetime.now(timezone.utc).isoformat()
    })

    save_state(state)

    print(f"Posted image: {chosen['name']}")
    print(f"Caption: {caption}")
    print(f"Telegram message ID: {new_message_id}")
    print("state.json updated.")

    return result


publication_result = publish_random_photo()


## Notes

- Add JPG, JPEG, PNG, or WEBP images to `images/`.
- Put one caption per line in `comments.txt`.
- The bot must be a channel administrator with permission to post and delete messages.
- GitHub Actions must pass `TELEGRAM_BOT_TOKEN`, `TELEGRAM_CHAT_ID`, `GITHUB_REPOSITORY`, and `GITHUB_REF_NAME` as environment variables.
- Keep each image below 10 MB.
